# Reproducing Peterson et al. (2021) from dandiset 000055

**Paper.** Peterson SM, Singh SH, Wang NXR, Rao RPN, Brunton BW (2021). Behavioral and neural
variability of naturalistic arm movements. *eNeuro* 8(3), ENEURO.0007-21.2021.
[doi:10.1523/ENEURO.0007-21.2021](https://doi.org/10.1523/ENEURO.0007-21.2021). The claims file was
written from the version of record in Europe PMC (PMC8225404): main text, Methods, legends, Tables 1
and 2 and the legends of the Extended Data figures.

**Dataset.** [DANDI:000055](https://dandiarchive.org/dandiset/000055/0.220127.0436) (AJILE12),
version 0.220127.0436. The paper cites no DANDI version: its data statement points to the curated
epochs on Figshare, and the dandiset was published later. This is the only published version, and
the draft lists the same 55 files with the same total size, so the published version is pinned.
The files are streamed; nothing is downloaded.

**What the dataset holds.** 12 subjects and 55 subject-days of preprocessed ECoG (500 Hz), pose of
nine key points (30 frames per second), 8088 contralateral wrist movement events with the paper's
ten features, coarse behaviour labels, and an electrode table with MNI positions, a `good` flag and
the paper's regression R2 per electrode. Sex and age are not in it.

**Outside input.** The projection of electrodes onto the eight AAL regions is the paper's own
output on Figshare ([doi:10.6084/m9.figshare.12114408.v1](https://doi.org/10.6084/m9.figshare.12114408.v1)),
used for the region claims (`data: with_inputs`) and checked by the link claim `roi-projection-link`.

**Event sets.** Reading a is every event in the files. Reading b drops ECoG segments with missing
samples, with power below 0 dB or with 115-125 Hz power more than 3 SD above the subject's segments
(Methods); the summary of segment power these tests use is an assumption (helper docstring). The
false positives the paper removed are not marked in the files, so they cannot be dropped.

| Claim | Tested here |
|---|---|
| subject-days, roi-projection-link (link claims) | tested |
| cohort, fig2-event-counts, fig2-features-differ-across-subjects, fig2-reach-shape, fig2-bimanual-skew | tested |
| extfig2-1c-contralateral-larger, extfig2-2-feature-correlations, fig2-days-and-waking | tested |
| fig3-group-lfb-decrease-hfb-increase, fig3-sensorimotor-largest, fig3-lfb-more-widespread | tested (the last blocked: its readings disagree) |
| fig4-precentral-subject-variability, fig5-band-power-differs-across-subjects, table1-subject-band-power-ci | tested |
| fig5-recording-day-effects, extfig5-1-baseline-reduces-day-variability, discussion-day-changes-small | tested |
| fig6b-best-models-sensorimotor, fig6-r2-at-most-025, fig6c-angle-and-day-most-informative, extfig6-2-angle-most-retained, fig6d-upward-reaches-enhance-pattern, fig6d-angle-and-day-largest-coefficients | blocked, too large (timed below) |

The notebook reads about 8000 ECoG segments of 10 s from 55 files of 8 to 20 GB each; that step takes
about half an hour.

In [ ]:
import os
import time
import warnings

os.environ["OMP_NUM_THREADS"] = "1"
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ajile

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
SUBJECTS = [f"{i:02d}" for i in range(1, 13)]

In [ ]:
started = time.time()
assets = ajile.list_assets()
sessions = ajile.load_sessions(assets)
# LOAD-BEH-END
print(f"{len(sessions)} files of dandiset {ajile.DANDISET_ID} version {ajile.DANDISET_VERSION}, "
      f"{assets.size_gb.sum():.0f} GB in all; metadata, events and wrist traces read in "
      f"{time.time() - started:.0f} s")

In [ ]:
# One table of events. Time of day from the midnight clock of the files (see the helper).
rows = []
for s in sessions:
    e = s["events"].copy()
    e["subject"], e["day"], e["k"] = s["subject"], s["day"], np.arange(len(e))
    rows.append(e)
ev = pd.concat(rows, ignore_index=True)
ev = ev.rename(columns={"Reach_magnitude_px": "magnitude", "Reach_angle_degrees": "angle_file",
                        "Onset_speed_px_per_sec": "speed", "Speech_ratio": "speech",
                        "Bimanual_ratio": "bimanual_ratio", "Bimanual_overlap": "bimanual_overlap",
                        "Bimanual_class": "bimanual_class"})
# The files' angle is atan2(dy, dx) in image coordinates (checked below); the paper's angle
# negates it (y up) and folds 90-270 degrees onto 90 to -90 (Methods; code).
ev["angle"] = ajile.fold_angle(-ev["angle_file"])
ev["hour"] = ev["start_time"] / 3600.0
ev["tod_bin"] = np.clip((ev["hour"] // 8).astype(int), 0, 2)
FEATURES = ["day", "hour", "duration", "magnitude", "angle", "speed", "speech",
            "bimanual_ratio", "bimanual_overlap", "bimanual_class"]
print(len(ev), "events")

## Data check

Counts by subject and day, units and ranges of every variable used, and the clocks of the
data streams. Each check of the metadata against the data is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "subject": s["subject"], "day": s["day"], "subject_id": s["subject_id"], "session_id": s["session_id"],
    "session_start_time": s["session_start_time"], "electrodes": len(s["electrodes"]),
    "good": int(s["electrodes"]["good"].sum()), "events": len(s["events"]), "wrist": s["reach_event_desc"],
    "ecog_hours": round(s["ecog_shape"][0] / s["ecog_rate"] / 3600, 2), "ecog_Hz": s["ecog_rate"],
    "pose_hours": round(s["pose_shape"][0] / s["pose_rate"] / 3600, 2), "pose_Hz": s["pose_rate"],
    "first_event_h": round(s["events"].start_time.min() / 3600, 2),
    "last_event_h": round(s["events"].start_time.max() / 3600, 2)} for s in sessions])
print(info.to_string(index=False))
print("\nSubjects:", info.subject.nunique(), "; files:", len(info), "; events:", int(info.events.sum()))
print("subject_id and session_id agree with the file names:",
      bool((info.subject_id == info.subject).all() and (info.session_id.astype(int) == info.day).all()))
print("Subject fields in the files:", sorted({tuple(s["subject_fields"]) for s in sessions}))
print("ECoG unit and conversion:", {(s["ecog_unit"], s["ecog_conversion"]) for s in sessions},
      "; pose unit:", {s["pose_unit"] for s in sessions})
LOC = ["x", "y", "z", "group_name", "low_freq_R2", "high_freq_R2"]
print("Electrode positions, groups and stored R2 identical across a subject's files:",
      all(sessions[i]["electrodes"][LOC].equals(sessions[j]["electrodes"][LOC])
          for i in range(len(sessions)) for j in range(len(sessions))
          if sessions[i]["subject"] == sessions[j]["subject"]))
print("Good flags differ between a subject's files:",
      sorted({s["subject"] for s in sessions for t in sessions
              if s["subject"] == t["subject"] and not np.array_equal(s["electrodes"].good.values, t["electrodes"].good.values)}))

The files hold 12 subjects and 55 subject-days, ECoG at 500 Hz and pose at 30 frames per
second. The ECoG is stored with a conversion of 1e-6 to volts, so the stored values are
microvolts; the pose is labelled meters but holds pixels (the data descriptor gives pixels, and
the values span hundreds). The subject has no sex or age in the files or the archive's metadata.
Three files hold less than 24 h of ECoG and start later (S01 day 7, S04 day 3, S06 day 3); the
clocks of these files are checked below.

In [ ]:
desc = {c: None for c in ["start_time", "stop_time", "duration", "magnitude", "angle_file", "angle", "speed",
                          "speech", "bimanual_ratio", "bimanual_overlap", "bimanual_class", "hour"]}
print(ev[list(desc)].describe().T[["count", "mean", "min", "50%", "max"]].round(3).to_string())
print("\nMissing values in any feature:", int(ev[list(desc)].isna().sum().sum()))
print("Bimanual ratio exactly 0.5 (the code's fill for missing values):", int((ev.bimanual_ratio == 0.5).sum()))
print("Bimanual overlap above 1 (the file calls it a fraction of the movement):",
      int((ev.bimanual_overlap > 1).sum()), "events")
print("Bimanual class values:", sorted(ev.bimanual_class.unique()))
print("Events per day, largest:", int(ev.groupby(["subject", "day"]).size().max()),
      "(Methods: up to 200 per day)")
print("Reach durations (stop - start) within 0.5-4 s (Methods' movement duration rule):",
      int(ev.duration.between(0.5, 4).sum()), "of", len(ev), "; shortest", round(ev.duration.min(), 3), "s")

In [ ]:
# Reach angle convention: angle and magnitude from the wrist trace (onset to the largest
# displacement within the reach), compared with the files' values.
rows = []
for s in sessions:
    w = "R_Wrist" if s["reach_event_desc"] == "r_wrist" else "L_Wrist"
    tr = s["wrist"][w]
    for k, d in enumerate(s["events"].duration.values):
        n = int(round(d * ajile.POSE_HZ))
        seg = tr[k, 45:46 + min(n, 45)]
        if np.isnan(seg).any():
            continue
        dd = seg - seg[0]
        r = np.linalg.norm(dd, axis=1)
        j = r.argmax()
        rows.append((s["events"]["Reach_angle_degrees"].values[k], np.degrees(np.arctan2(dd[j, 1], dd[j, 0])),
                     s["events"]["Reach_magnitude_px"].values[k], r[j], d <= 1.5))
ang = pd.DataFrame(rows, columns=["file_angle", "trace_angle", "file_mag", "trace_mag", "within_window"])
circ = lambda a, b: np.abs((a - b + 180) % 360 - 180)
a = ang[ang.within_window]
print(f"{len(a)} reaches of at most 1.5 s with a full trace")
print("median |file angle - trace angle|, image coordinates:", round(np.median(circ(a.file_angle, a.trace_angle)), 2),
      "deg; with the trace angle negated (y up):", round(np.median(circ(a.file_angle, -a.trace_angle)), 2), "deg")
print("median file magnitude / trace magnitude:", round(np.median(a.file_mag / a.trace_mag), 3))
print("Folded angle (paper convention, +90 up): share of events above 0:", round((ev.angle > 0).mean(), 3))

The files' reach angle is the angle of the reach in image coordinates (y pointing down), not yet
negated, so upward reaches have negative values; the paper's angle (+90 for upward) is its
negation folded onto -90 to 90, as the code does. The files do not say which convention they use.
The magnitudes match the traces.

In [ ]:
# Clock of the pose and of the event times: contralateral wrist displacement from its mean
# position over -1.5 to -1 s, at the events and at the events shifted by -30 s.
def median_disp(tr, base_slice=slice(0, 16)):
    base = np.nanmean(tr[:, base_slice], 1)
    return np.nanmedian(np.linalg.norm(tr - base[:, None], axis=2), 0)

rows = []
for s in sessions:
    w = "R_Wrist" if s["reach_event_desc"] == "r_wrist" else "L_Wrist"
    m = median_disp(s["wrist"][w])
    rows.append({"subject": s["subject"], "day": s["day"], "disp_at_-1s": m[15].round(1),
                 "disp_at_0s": m[45].round(1), "disp_at_+1s": m[75].round(1)})
clk = pd.DataFrame(rows)
print(clk.to_string(index=False))
print("Sessions where the contralateral wrist moves less than 10 pixels by +1 s:",
      clk[clk["disp_at_+1s"] < 10][["subject", "day"]].values.tolist())

In every file the contralateral wrist is still before the event time and moves after it, so the
pose and the event table share a clock, including in the three files with less than 24 h of
ECoG (whose events lie between 8 and 24 h). The shifted control needs the ECoG and is below.

In [ ]:
# Time of day: the coarse behaviour labels by hour. Sleep should fall at night if the files
# start at midnight, as the data descriptor says.
lab = pd.concat([s["epochs"].assign(subject=s["subject"]) for s in sessions])
lab["hour"] = ((lab.start + lab.stop) / 2 / 3600).astype(int)
sleep = lab[lab.label == "Sleep/rest"].groupby("hour").size().reindex(range(24), fill_value=0)
print("Sleep/rest labels by hour of the file clock:\n", sleep.to_string())
print("Share of Sleep/rest labels at 0-8 h:", round(sleep[:8].sum() / sleep.sum(), 3),
      "; events at 0-8 h:", round((ev.hour < 8).mean(), 3))
print("session_start_time values:", sorted(info.session_start_time.unique()))

Sleep labels fall mostly in the first hours of the file clock, which fits files that start at
midnight local time. The files' `session_start_time` of 19:00 at UTC-5 is midnight UTC, so the
stated time zone is not the local one; the three late files have start times of 03:07 to 03:31
at UTC-5, the start of their ECoG (checked below). The hour of an event is taken from the file
clock.

In [ ]:
# Duplicates: identical event tables or wrist traces in two files.
keys = {}
for s in sessions:
    k1 = np.round(s["events"].start_time.values, 3).tobytes()
    k2 = np.nan_to_num(s["wrist"]["R_Wrist"]).round(2).tobytes()
    keys.setdefault(("events", hash(k1)), []).append((s["subject"], s["day"]))
    keys.setdefault(("wrist", hash(k2)), []).append((s["subject"], s["day"]))
dups = {k: v for k, v in keys.items() if len(v) > 1}
print("Files with identical event times or identical wrist traces:", dups if dups else "none")

## ECoG: projection matrices and event segments

The paper's projection of electrodes onto the eight regions (AAL atlas through the Measure
Projection Toolbox) is taken from its own output on Figshare (doi:10.6084/m9.figshare.12114408.v1),
which the claims file names as the paper's output of that step. Claim `roi-projection-link`
checks that its rows can be matched to the dataset's electrodes. The bad electrodes are those of
each subject's first file, as the paper's code takes them.

In [ ]:
roi_meta = ajile.roi_files()
roi_mats, weights_check = {}, {}
for sub in SUBJECTS:
    roi_mats[sub] = ajile.read_roi_matrix(sub, roi_meta)
first_good = {}
for s in sessions:
    first_good.setdefault(s["subject"], s["electrodes"]["good"].values.copy())
for s in sessions:
    s["subject_good"] = first_good[s["subject"]]
print("Figshare files read (name, md5):")
for sub in SUBJECTS:
    print(f"  aal_{sub}_elecs2ROI.csv", roi_meta[f"aal_{sub}_elecs2ROI.csv"][1])

In [ ]:
started = time.time()
weights = {sub: ajile.roi_weights(roi_mats[sub], first_good[sub]) for sub in SUBJECTS}
ecog = ajile.load_ecog(ajile.ecog_jobs(sessions, weights))
# LOAD-ECOG-END
print(f"ECoG segments of {sum(sum(r['ok']) for r in ecog)} events read and reduced in "
      f"{(time.time() - started) / 60:.1f} min")

In [ ]:
# Per-event ECoG measures, aligned with the event table.
REG = {n: i for i, n in enumerate(ajile.ROI_NAMES)}
ev["ok"] = np.concatenate([r["ok"] for r in ecog])
for key in ["q_all", "q_hf", "q_log"]:
    ev[key] = np.concatenate([r[key] for r in ecog])
spec = np.full((len(ev), 8, len(ajile.FREQS), len(ajile.TIMES)), np.nan, dtype=np.float32)
i = 0
for r in ecog:
    for ok, sp in zip(r["ok"], r["roi_spec"]):
        if ok:
            spec[i] = sp
        i += 1
for kind in ["roi_band", "roi_band_raw"]:
    for meas in ["lfb_mean", "lfb_median", "hfb_mean", "hfb_median"]:
        vals = np.full((len(ev), 8), np.nan)
        i = 0
        for r in ecog:
            for ok, b in zip(r["ok"], r[kind]):
                if ok:
                    vals[i] = b[meas]
                i += 1
        for reg, j in REG.items():
            ev[f"{kind}:{reg}:{meas}"] = vals[:, j]
# Bad segments (Methods): power below 0 dB, or 115-125 Hz power more than 3 SD above the mean
# of the subject's segments. False positives are not marked in the files (see the data check).
thr = ev[ev.ok].groupby("subject").q_hf.agg(lambda v: v.mean() + 3 * v.std())
ev["bad_missing"] = ev.ok & ~np.isfinite(ev.q_all)          # segments with missing (NaN) samples
ev["bad_low"] = ev.q_all < 0
ev["bad_hf"] = ev.q_hf > ev.subject.map(thr)
ev["keep_b"] = ev.ok & ~ev.bad_missing & ~ev.bad_low & ~ev.bad_hf
TIMES, FREQS = ajile.TIMES, ajile.FREQS
print("Events with a full ECoG segment:", int(ev.ok.sum()), "of", len(ev))
print("Segments with missing (NaN) samples:", int(ev.bad_missing.sum()), "in",
      ev[ev.bad_missing].groupby(["subject", "day"]).size().to_dict())
print("Segments below 0 dB:", int(ev.bad_low.sum()), "; 115-125 Hz above 3 SD:", int((ev.bad_hf & ~ev.bad_low).sum()),
      "; kept (reading b):", int(ev.keep_b.sum()))
print("Other reading of the 0 dB test (mean of the log density below 0 dB):", int((ev.q_log < 0).sum()), "of",
      int(ev.ok.sum()), "segments would go, so that reading is not usable")
print(ev[ev.ok].groupby("subject")[["q_all", "q_hf"]].agg(["min", "median", "max"]).round(1).to_string())

In [ ]:
# segment_power against MNE's own Morlet transform, on one segment of the data.
from mne.time_frequency import tfr_array_morlet
import h5py, remfile
s0 = sessions[0]
f0 = h5py.File(remfile.File(ajile.asset_url(s0["asset_id"])), "r")
c0 = int(round(s0["events"].start_time.values[0] * ajile.FS))
x0 = f0["acquisition/ElectricalSeries/data"][c0 - 2500:c0 + 2500, :8].T.astype(float)
f0.close()
mine = ajile.segment_power(x0)
ref = tfr_array_morlet(x0[None], sfreq=ajile.FS, freqs=FREQS, n_cycles=FREQS / 4, zero_mean=True, use_fft=False,
                       decim=ajile.DECIM, output="power", verbose=False)[0][..., 5:96]
print("largest difference from MNE tfr_array_morlet(use_fft=False), dB:", float(np.abs(mine - 10 * np.log10(ref)).max()))
print("segment amplitude range (microvolts):", round(float(x0.min()), 1), round(float(x0.max()), 1))

In [ ]:
# Clock of the ECoG against the events: precentral and postcentral band power change from the
# -1.5 to -1 s baseline to 0-0.5 s, at the event time and at a pseudo-event 3 s earlier
# (-3 to -2.5 s against -4.5 to -4 s) in the same segments; median over events per file.
ti = lambda lo, hi: np.nonzero((TIMES >= lo - 1e-6) & (TIMES <= hi + 1e-6))[0]
lf = np.nonzero((FREQS >= 8) & (FREQS <= 32))[0]
hf = np.nonzero((FREQS >= 76) & (FREQS <= 100))[0]
def band_change(sp, band, t_on, t_base):
    return sp[..., band, :][..., t_on].mean((-1, -2)) - sp[..., band, :][..., t_base].mean((-1, -2))
rows = []
for (sub, day), g in ev[ev.keep_b].groupby(["subject", "day"]):
    sp = spec[g.index][:, [REG["precentral"], REG["postcentral"]]]
    row = {"subject": sub, "day": day, "n": len(g)}
    for name, band in [("HFB", hf), ("LFB", lf)]:
        true = band_change(sp, band, ti(0, 0.5), ti(-1.5, -1))
        pseudo = band_change(sp, band, ti(-3, -2.5), ti(-4.5, -4))
        best = np.argmax(np.abs(np.median(true, 0)))
        row[f"{name}_event"] = round(float(np.median(true[:, best])), 2)
        row[f"{name}_pseudo"] = round(float(np.median(pseudo[:, best])), 2)
    rows.append(row)
clock = pd.DataFrame(rows)
print(clock.to_string(index=False))
clock_ok = (clock.HFB_event > 0) & (clock.LFB_event < 0) & (clock.HFB_event > clock.HFB_pseudo) & (clock.LFB_event < clock.LFB_pseudo)
print("Files with HFB up, LFB down at the event and both larger than at the pseudo-event:", int(clock_ok.sum()), "of", len(clock))
print("Files where this does not hold:", clock[~clock_ok][["subject", "day"]].values.tolist())

In [ ]:
# The three files with less than 24 h of ECoG: event times on the ECoG clock with the offset
# of the helper (ECoG ends at midnight) against no offset (ECoG starts at midnight).
late = [s for s in sessions if ajile.ecog_offset(s) > 1]
alt = ajile.load_ecog(ajile.ecog_jobs(late, weights, offset=False), workers=3)
for s, r in zip(late, alt):
    g = ev[(ev.subject == s["subject"]) & (ev.day == s["day"]) & ev.keep_b]
    pre = REG["precentral"]
    a_h = band_change(spec[g.index][:, pre], hf, ti(0, 0.5), ti(-1.5, -1))
    a_l = band_change(spec[g.index][:, pre], lf, ti(0, 0.5), ti(-1.5, -1))
    sp_b = np.array([x[pre] for x, ok in zip(r["roi_spec"], r["ok"]) if ok and np.isfinite(x[pre]).all()])
    b_h = band_change(sp_b, hf, ti(0, 0.5), ti(-1.5, -1)) if len(sp_b) else np.array([np.nan])
    b_l = band_change(sp_b, lf, ti(0, 0.5), ti(-1.5, -1)) if len(sp_b) else np.array([np.nan])
    print(f"S{s['subject']} day {s['day']}: offset {ajile.ecog_offset(s):.1f} s; with offset {len(a_h)} events, "
          f"precentral HFB {np.median(a_h):+.2f} dB, LFB {np.median(a_l):+.2f} dB; without offset {len(sp_b)} events in "
          f"range, HFB {np.median(b_h):+.2f} dB, LFB {np.median(b_l):+.2f} dB; ReachEvents minus offset-corrected "
          f"times, max |diff| {np.max(np.abs(s['reach_event_times'] - (s['events'].start_time.values - ajile.ecog_offset(s)))):.3f} s")

## Claim `subject-days`

> The dataset's subjects and recording days are the paper's subjects S01 to S12 and the
> recording days the paper lists for each.

In [ ]:
TABLE2 = {"01": [3, 4, 5, 7], "02": [3, 4, 5, 6], "03": [3, 4, 5, 6], "05": [3, 4, 7]}
TABLE2 = {s: TABLE2.get(s, [3, 4, 5, 6, 7]) for s in SUBJECTS}
days = info.groupby("subject").day.apply(list).to_dict()
for sub in SUBJECTS:
    print(f"S{sub}: dataset days {days[sub]}, Table 2 {TABLE2[sub]}, match {days[sub] == TABLE2[sub]}")
print("All 12 match:", all(days[s] == TABLE2[s] for s in SUBJECTS), "; subject-days:", sum(len(v) for v in days.values()))

**Result: reproduced.** The dataset's subjects 01 to 12 have exactly the days of Table 2 (55 subject-days), so the subjects are matched by number.

## Claim `roi-projection-link`

> The rows of the paper's projection matrices (Figshare) can be matched one to one to the
> dataset's electrodes, and the weights fit the electrodes' positions.

In [ ]:
cent_rows = {}
rows = []
rng_link = np.random.default_rng(55)
for sub in SUBJECTS:
    df = roi_mats[sub]
    el = [s for s in sessions if s["subject"] == sub][0]["electrodes"]
    n_rows = len(df) - 1
    zero = np.nonzero(df.iloc[1:].transpose().mean().values == 0)[0]
    cen = pd.read_csv(roi_meta[f"aal_{sub}_ROIcentroids_Lside.csv"][0])
    cen = cen.iloc[[df.columns.get_loc(c) for c in ajile.ROI_COLUMNS]].to_numpy(float)
    w = weights[sub]
    xyz = el[["x", "y", "z"]].to_numpy(float).copy()
    xyz[:, 0] = -np.abs(xyz[:, 0])                       # right hemisphere mirrored to the left
    dist = np.linalg.norm(xyz[:, None, :] - cen[None], axis=2)
    use = w.sum(1) > 0
    agree = np.mean(np.argmax(w[use], 1) == np.argmin(dist[use], 1))
    null = np.mean([np.mean(np.argmax(w[use], 1) == np.argmin(dist[use][rng_link.permutation(use.sum())], 1))
                    for _ in range(200)])
    rho = stats.spearmanr(w[use].ravel(), -dist[use].ravel())[0]
    rows.append({"subject": sub, "electrodes": len(el), "matrix_rows": n_rows, "all_zero_rows": len(zero),
                 "zero_row_groups": ",".join(sorted(set(el.group_name.values[zero]))) if n_rows == len(el) else
                 f"rows {zero.min()}-{zero.max()} (beyond the {len(el)} electrodes)",
                 "rows_used": len(w), "top_weight_is_nearest_centroid": round(agree, 3),
                 "same_with_order_shuffled": round(null, 3), "spearman_weight_vs_proximity": round(rho, 3)})
link = pd.DataFrame(rows)
print(link.to_string(index=False))
print("Subjects whose rows match the electrodes one to one:", int((link.rows_used == link.electrodes).sum()), "of 12")
print("Subjects where the weights fit positions better than with the order shuffled:",
      int((link.top_weight_is_nearest_centroid > link.same_with_order_shuffled).sum()), "of 12")
dens = pd.concat([roi_mats[s].iloc[0] for s in SUBJECTS], axis=1).mean(1)
print("Regions with mean electrode density > 3 (Methods):",
      [c for c in dens.index if dens[c] > 3], "\nthe eight of the claims file:", ajile.ROI_COLUMNS)

**Result: reproduced.** For all 12 subjects the projection matrix has one row per electrode of the NWB file (S01 has four extra all-zero rows at the end, dropped by the code's rule), the all-zero rows of S03 and S12 fall on depth electrode groups (LAD and LPD; AID), and the region with the largest weight is the region with the nearest centroid for 0.59 to 0.79 of electrodes, against 0.16 to 0.26 with the electrode order shuffled; the Spearman correlation of weight with proximity is 0.92 to 0.94. The regions with mean density above 3 are the eight of the claims file.

## Claim `cohort`

> Twelve subjects (8 male, 4 female, 29.4 +- 7.9 years) had on average 87.0 +- 12.9 surface
> electrodes; five also had 23.2 +- 12.1 depth electrodes; five were implanted in the right
> hemisphere and seven in the left.

In [ ]:
rows = []
for sub in SUBJECTS:
    s = [x for x in sessions if x["subject"] == sub][0]
    el = s["electrodes"]
    depth_groups = [g for g, d in s["group_desc"].items() if "depth" in d.lower()]
    is_depth = el.group_name.isin(depth_groups)
    rows.append({"subject": sub, "surface": int((~is_depth).sum()), "depth": int(is_depth.sum()),
                 "depth_groups": ",".join(depth_groups), "wrist": s["reach_event_desc"],
                 "mean_x": round(el.x.mean(), 1)})
coh = pd.DataFrame(rows)
print(coh.to_string(index=False))
hemi_right = (coh.wrist == "l_wrist").sum()
print(f"\nsubjects {len(coh)}; surface electrodes mean {coh.surface.mean():.1f} +- {coh.surface.std():.1f} (SD); "
      f"subjects with depth electrodes {(coh.depth > 0).sum()}, their mean {coh[coh.depth > 0].depth.mean():.1f} +- "
      f"{coh[coh.depth > 0].depth.std():.1f}; right hemisphere (left wrist events, x > 0) {hemi_right}")
print("Right hemisphere by wrist agrees with the sign of mean x:", bool(((coh.wrist == "l_wrist") == (coh.mean_x > 0)).all()))
print("Sex and age: not in the files or the archive's metadata (subject fields:", sessions[0]["subject_fields"], ")")
both = coh.copy()
both["all"] = both.surface + both.depth
print("Variant, every electrode counted: mean", round(both["all"].mean(), 1))

**Result: partially reproduced.** 12 subjects, a mean of 84.8 surface electrodes (87.0 +- 12.9), a mean of 23.7 depth electrodes in the subjects that have them (23.2 +- 12.1) and 5 right-hemisphere subjects match. Seven subjects have electrode groups described as depth in the files, where the paper says five. Sex and age are not in the dataset, so those two values are not testable.

## Claim `fig2-event-counts`

> Each subject contributed between 151 and 947 movement events (median 640 across subjects).

In [ ]:
cnt = pd.DataFrame({"a_all_events": ev.groupby("subject").size(),
                    "b_bad_segments_dropped": ev[ev.keep_b].groupby("subject").size()})
print(cnt.T.to_string())
for c in cnt:
    v = cnt[c]
    print(f"{c}: fewest {v.min()} (151 +- 15.1), most {v.max()} (947 +- 94.7), median {v.median():.1f} (640 +- 64); "
          f"within: {abs(v.min() - 151) <= 15.1}, {abs(v.max() - 947) <= 94.7}, {abs(v.median() - 640) <= 64}")
print("Total events:", len(ev), "; the data descriptor (Peterson et al. 2022, Sci Data) states 6931 verified events")

**Result: not reproduced.** The dataset holds 211 to 996 events per subject (median 770.5); after dropping segments with missing samples, below 0 dB or with high 115-125 Hz power, 211 to 979 (median 739). The fewest (211 against 151 +- 15.1) and the median (against 640 +- 64) are outside tolerance under both readings; the most is within. The files hold 8088 events, where the data descriptor states 6931, and no event is marked as a false positive.

## Claim `fig2-features-differ-across-subjects`

> All ten behavioural and environmental features differ significantly across subjects.

In [ ]:
rows = []
for label, sel in [("b", ev.keep_b), ("a", np.ones(len(ev), bool))]:
    for f in FEATURES:
        groups = [g[f].values for _, g in ev[sel].groupby("subject")]
        h, p = stats.kruskal(*groups)
        rows.append({"reading": label, "feature": f, "H": round(h, 1), "p": p, "p_below_0001": p < 0.001})
kw = pd.DataFrame(rows)
print(kw.to_string(index=False))
print("All ten p < 0.001, reading b:", bool(kw[kw.reading == "b"].p_below_0001.all()),
      "; reading a:", bool(kw[kw.reading == "a"].p_below_0001.all()))
print("Largest p:", kw.groupby("reading").p.max().to_dict())

**Result: reproduced.** All ten Kruskal-Wallis tests across subjects give p < 0.001 under both event sets (largest p 9.9e-35).

## Claim `fig2-reach-shape`

> Reach duration and reach magnitude vary little across subjects, with most reaches lasting
> under 2 s and covering under 200 pixels, and reach angle distributions tend to be bimodal with
> peaks near +90 and -90 degrees, with few side-to-side reaches.

In [ ]:
from scipy.stats import gaussian_kde
grid = np.linspace(-90, 90, 361)
def bimodal(a):
    dens = gaussian_kde(a, bw_method="scott")(grid)
    peaks = [i for i in range(1, len(grid) - 1) if dens[i] > dens[i - 1] and dens[i] >= dens[i + 1]]
    # ends of the range count as maxima when the density rises towards them
    if dens[0] > dens[1]:
        peaks.append(0)
    if dens[-1] > dens[-2]:
        peaks.append(len(grid) - 1)
    top = sorted(peaks, key=lambda i: -dens[i])[:2]
    locs = sorted(grid[top])
    zero = dens[np.argmin(np.abs(grid))]
    ok = len(locs) == 2 and abs(locs[0] + 90) <= 15 and abs(locs[1] - 90) <= 15 and zero < dens[top].min()
    return ok, locs, zero / dens[top].min() if len(top) else np.nan
E = ev[ev.keep_b]
rows = []
for sub, g in list(E.groupby("subject")) + [("pooled", E)]:
    ok, locs, rel = bimodal(g.angle.values)
    rows.append({"subject": sub, "n": len(g), "median_duration_s": round(g.duration.median(), 2),
                 "share_under_2s": round((g.duration < 2).mean(), 3), "median_magnitude_px": round(g.magnitude.median(), 1),
                 "share_under_200px": round((g.magnitude < 200).mean(), 3), "angle_peaks": np.round(locs, 1).tolist(),
                 "density_at_0_over_lower_peak": round(rel, 2), "bimodal_at_pm90": ok,
                 "share_within_30deg_of_0": round((g.angle.abs() < 30).mean(), 3)})
rs = pd.DataFrame(rows)
print(rs.to_string(index=False))
subj = rs[rs.subject != "pooled"]
pool = rs[rs.subject == "pooled"].iloc[0]
dm = (subj.share_under_2s > 0.5) & (subj.share_under_200px > 0.5)
print(f"\nduration and magnitude part: pooled {pool.share_under_2s > 0.5 and pool.share_under_200px > 0.5}, "
      f"subjects {int(dm.sum())} of 12")
print(f"angle part: pooled {pool.bimodal_at_pm90}, subjects {int(subj.bimodal_at_pm90.sum())} of 12")
print("Variant, reading a (all events):")
E2 = ev
print(" share under 2 s", round((E2.duration < 2).mean(), 3), "; under 200 px", round((E2.magnitude < 200).mean(), 3),
      "; pooled angle bimodal at +-90:", bimodal(E2.angle.values)[0], np.round(bimodal(E2.angle.values)[1], 1).tolist())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for sub, g in E.groupby("subject"):
    axes[0].hist(g.duration, bins=np.linspace(0, 4, 41), histtype="step", density=True)
    axes[1].hist(g.magnitude, bins=np.linspace(0, 700, 71), histtype="step", density=True)
    axes[2].plot(grid, gaussian_kde(g.angle, bw_method="scott")(grid), lw=1)
axes[0].set_xlabel("reach duration (s)"); axes[1].set_xlabel("reach magnitude (pixels)"); axes[2].set_xlabel("reach angle (deg, +90 up)")
plt.tight_layout(); plt.show()
print("Plotted ranges: duration", round(E.duration.min(), 2), "-", round(E.duration.max(), 2), "s; magnitude",
      round(E.magnitude.min(), 1), "-", round(E.magnitude.max(), 1), "px (none outside the axes)")

**Result: partially reproduced.** More than half of the reaches last under 2 s and cover under 200 pixels pooled (0.918 and 0.915) and in all 12 subjects. The angle part does not hold: the two highest maxima of the angle density lie at -71.5 and 75.0 degrees pooled, and within 15 degrees of +-90 in no subject (S02 and S04 have a single maximum, and in S03, S06 and S07 the density at 0 exceeds the lower peak); 22.9% of reaches lie within 30 degrees of horizontal.

## Claim `fig2-bimanual-skew`

> Bimanual ratio and bimanual overlap are skewed toward unimanual movement of the contralateral
> arm, less so for subjects 02, 04 and 05, while the bimanual class is mostly bimanual.

In [ ]:
def bim_table(E):
    rows = []
    for sub, g in list(E.groupby("subject")) + [("pooled", E)]:
        rows.append({"subject": sub, "median_ratio": round(g.bimanual_ratio.median(), 3),
                     "ratio_skewed": g.bimanual_ratio.median() < 0.5,
                     "overlap_skewness": round(stats.skew(g.bimanual_overlap), 3),
                     "overlap_skewed": stats.skew(g.bimanual_overlap) > 0,
                     "share_class_bimanual": round(g.bimanual_class.mean(), 3),
                     "class_mostly_bimanual": g.bimanual_class.mean() > 0.5})
    return pd.DataFrame(rows)
for label, E_ in [("b", ev[ev.keep_b]), ("a", ev)]:
    bt = bim_table(E_)
    subj = bt[bt.subject != "pooled"]
    pool = bt[bt.subject == "pooled"].iloc[0]
    top_ratio = subj.sort_values("median_ratio", ascending=False).subject.head(5).tolist()
    low_skew = subj.sort_values("overlap_skewness").subject.head(5).tolist()
    named = all(s in top_ratio for s in ["02", "04", "05"]) and all(s in low_skew for s in ["02", "04", "05"])
    print(f"Reading {label}")
    print(bt.to_string(index=False))
    print(f"ratio part: pooled {pool.ratio_skewed}, subjects {int(subj.ratio_skewed.sum())}/12; overlap part: pooled "
          f"{pool.overlap_skewed}, subjects {int(subj.overlap_skewed.sum())}/12; class part: pooled {pool.class_mostly_bimanual}, "
          f"subjects {int(subj.class_mostly_bimanual.sum())}/12")
    print(f"five highest median ratios {top_ratio}; five lowest overlap skewness {low_skew}; S02, S04, S05 in both: {named}\n")

**Result: reproduced.** The median bimanual ratio is below 0.5 and the overlap skewness positive pooled and in all 12 subjects, the class is bimanual for more than half of the events pooled (0.725) and in all 12, and S02, S04 and S05 are among the five subjects with the highest median ratio and among the five with the lowest overlap skewness, under both event sets.

## Claim `extfig2-1c-contralateral-larger`

> Across all 12 subjects, the median displacement of the contralateral wrist after movement
> onset is substantially greater than that of the ipsilateral wrist, confirming that event
> detection mainly picked contralateral movements.

In [ ]:
t_pose = np.round(np.arange(91) / ajile.POSE_HZ + ajile.POSE_WIN[0], 3)
post = (t_pose >= 0) & (t_pose <= 1.5)
def disp(tr, reading):
    base = np.nanmean(tr[:, :16], 1) if reading == "a" else tr[:, 45]   # -1.5 to -1 s, or onset
    return np.linalg.norm(tr - base[:, None], axis=2)
rows = []
for sub in SUBJECTS:
    ss = [s for s in sessions if s["subject"] == sub]
    contra = "R_Wrist" if ss[0]["reach_event_desc"] == "r_wrist" else "L_Wrist"
    ipsi = "L_Wrist" if contra == "R_Wrist" else "R_Wrist"
    keep = np.concatenate([ev[(ev.subject == sub) & (ev.day == s["day"])].keep_b.values for s in ss])
    row = {"subject": sub, "contra": contra}
    for reading in ["a", "b"]:
        c = np.concatenate([disp(s["wrist"][contra], reading) for s in ss])[keep]
        i_ = np.concatenate([disp(s["wrist"][ipsi], reading) for s in ss])[keep]
        cm, im = np.nanmedian(c, 0)[post].max(), np.nanmedian(i_, 0)[post].max()
        row[f"{reading}_contra_peak_px"] = round(cm, 1)
        row[f"{reading}_ipsi_peak_px"] = round(im, 1)
        row[f"{reading}_ratio"] = round(cm / im, 2)
    rows.append(row)
cw = pd.DataFrame(rows)
print(cw.to_string(index=False))
for r in ["a", "b"]:
    print(f"reading {r}: contralateral peak larger in {(cw[f'{r}_contra_peak_px'] > cw[f'{r}_ipsi_peak_px']).sum()} of 12; "
          f"ratio min {cw[f'{r}_ratio'].min()}, median {cw[f'{r}_ratio'].median()}")

**Result: reproduced.** The peak of the median contralateral displacement exceeds the ipsilateral one in all 12 subjects under both readings of the rest position (ratios 1.12 to 37.8 under reading a, 1.08 to 42.0 under reading b). In S04 and S05 the two are close (ratios 1.08 to 1.32).

## Claim `extfig2-2-feature-correlations`

> Over all events, reach magnitude correlates with reach duration (r = 0.26) and onset speed
> (r = 0.56), reach duration with bimanual overlap (r = 0.48), and the bimanual features with
> one another (overlap and ratio 0.51, class and ratio 0.50, overlap and class 0.61).

In [ ]:
PAIRS = [("magnitude", "duration", 0.26), ("magnitude", "speed", 0.56), ("duration", "bimanual_overlap", 0.48),
         ("bimanual_overlap", "bimanual_ratio", 0.51), ("bimanual_class", "bimanual_ratio", 0.50),
         ("bimanual_overlap", "bimanual_class", 0.61)]
rows = []
for label, E_ in [("b", ev[ev.keep_b]), ("a", ev)]:
    corr = E_[FEATURES].corr()
    for x, y, paper in PAIRS:
        r = corr.loc[x, y]
        rows.append({"reading": label, "pair": f"{x} - {y}", "r": round(r, 3), "paper": paper,
                     "within_10pct": abs(r - paper) <= 0.1 * paper + 1e-9})
cr = pd.DataFrame(rows)
print(cr.to_string(index=False))
print("Within tolerance:", cr.groupby("reading").within_10pct.sum().to_dict(), "of 6")
print("\nPer subject, reading b (r):")
print(pd.DataFrame({f"{x}-{y}": ev[ev.keep_b].groupby("subject").apply(lambda g: g[x].corr(g[y])).round(2)
                    for x, y, _ in PAIRS}).to_string())

**Result: partially reproduced.** Five of the six correlations are within tolerance (0.604, 0.503, 0.502, 0.494, 0.599); reach magnitude and duration give r = 0.233, just below the tolerance of 0.26 +- 0.026, under both event sets (0.231 with all events).

## Claim `fig2-days-and-waking`

> Reach counts were fairly even across recording days except for subjects 04, 05 and 09, each
> with one day holding most events, and events occurred mostly during waking hours.

In [ ]:
for label, E_ in [("b", ev[ev.keep_b]), ("a", ev)]:
    per_day = E_.groupby(["subject", "day"]).size().unstack(fill_value=0)
    busiest = (per_day.max(1) / per_day.sum(1)).round(3)
    awake = E_.groupby("subject").hour.apply(lambda h: (h >= 8).mean()).round(3)
    out = pd.DataFrame({"busiest_day_share": busiest, "share_08_24h": awake})
    print(f"Reading {label}\n", per_day.to_string(), "\n", out.T.to_string())
    over = out.index[out.busiest_day_share > 0.5].tolist()
    print(f"subjects whose busiest day holds more than half: {over}; named 04, 05, 09; "
          f"waking part (more than half at 08-24 h) holds in {(out.share_08_24h > 0.5).sum()} of 12\n")

**Result: partially reproduced.** One day holds more than half of the events for S05 (0.605) and S09 (0.517) and for no other subject, but not for S04, whose busiest day holds 0.379. More than half of the events fall at 08 to 24 h in all 12 subjects (0.641 to 0.964), so the waking part holds.

## Claim `fig3-group-lfb-decrease-hfb-increase`

> At the group level, low-frequency power decreases and high-frequency power increases at
> movement initiation in several cortical regions, as in controlled movement experiments.

In [ ]:
rng_fig3 = np.random.default_rng(3)
keep = ev.keep_b.values
subj_med = np.stack([np.median(spec[np.nonzero(keep & (ev.subject.values == sub))[0]], axis=0) for sub in SUBJECTS])
base_i = ti(-1.5, -1)
group = subj_med.mean(0)
subj_c = subj_med - np.median(group[..., base_i], axis=-1, keepdims=True)[None]     # plot_specs_group
masked = np.stack([ajile.boot_sig_mask(np.moveaxis(subj_c[:, r], 0, -1), base_i, rng_fig3) for r in range(8)])
unmasked = subj_c.mean(0)
LF34 = np.nonzero((FREQS >= 4) & (FREQS <= 30))[0]
HF50 = np.nonzero((FREQS >= 50) & (FREQS <= 120))[0]
t_a, t_b = ti(0, 0.5), np.array([np.argmin(np.abs(TIMES))])
rows = []
for r, name in enumerate(ajile.ROI_NAMES):
    row = {"region": name}
    for lab, tt in [("a_0-0.5s", t_a), ("b_t0", t_b)]:
        row[f"{lab}_LF4-30"] = round(float(masked[r][LF34][:, tt].mean()), 3)
        row[f"{lab}_HF50-120"] = round(float(masked[r][HF50][:, tt].mean()), 3)
    row["unmasked_a_LF"] = round(float(unmasked[r][LF34][:, t_a].mean()), 3)
    row["unmasked_a_HF"] = round(float(unmasked[r][HF50][:, t_a].mean()), 3)
    row["significant_bins"] = int((masked[r] != 0).sum())
    rows.append(row)
f3 = pd.DataFrame(rows)
print(f3.to_string(index=False))
named = ["precentral", "postcentral", "inferior parietal"]
for lab in ["a_0-0.5s", "b_t0"]:
    holds = f3[f3.region.isin(named)].apply(lambda r: r[f"{lab}_LF4-30"] < 0 and r[f"{lab}_HF50-120"] > 0, axis=1)
    print(f"reading {lab}: pattern holds in {int(holds.sum())} of 3 named regions",
          dict(zip(named, holds.values)))
print("Regions with the pattern under reading a:",
      f3[(f3["a_0-0.5s_LF4-30"] < 0) & (f3["a_0-0.5s_HF50-120"] > 0)].region.tolist())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5), sharex=True, sharey=True)
tsel = ti(-1.5, 1.5)
for r, ax in enumerate(axes.ravel()):
    im = ax.pcolormesh(TIMES[tsel], FREQS, masked[r][:, tsel], cmap="RdBu_r", vmin=-1, vmax=1, shading="nearest")
    ax.axvline(0, color="k", lw=1)
    ax.set_title(ajile.ROI_NAMES[r], fontsize=9)
axes[1, 0].set_xlabel("time (s)"); axes[1, 0].set_ylabel("frequency (Hz)")
fig.colorbar(im, ax=axes, label="dB (masked)")
plt.show()
print("masked values shown, -1.5 to 1.5 s: range", round(float(masked[..., tsel].min()), 2), "to",
      round(float(masked[..., tsel].max()), 2), "dB; values beyond +-1 dB:", int((np.abs(masked[..., tsel]) > 1).sum()))

**Result: reproduced.** After masking, the mean over 4-30 Hz is negative and over 50-120 Hz positive in precentral (-0.885 and 0.496 dB), postcentral (-0.890, 0.507) and inferior parietal (-0.663, 0.277) over 0 to 0.5 s, and also at the time bin of onset (reading b). Middle frontal and supramarginal show the same pattern.

## Claim `fig3-sensorimotor-largest`

> Spectral power changes are largest in precentral, postcentral and inferior parietal regions,
> and the three temporal regions show minimal movement-related changes.

In [ ]:
temporal = ["superior temporal", "middle temporal", "inferior temporal"]
def check_rank(vals):
    order = list(pd.Series(vals).sort_values(ascending=False).index)
    top3 = set(order[:3]) == set(named)
    temp_below = all(vals[t] < min(vals[n] for n in named) for t in temporal)
    return order, top3, temp_below, sum(n in order[:3] for n in named)
res = {}
va = {n: float(np.abs(masked[r][:, t_a]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
vc = {n: float(np.abs(masked[r][:, ti(-0.5, 0.5)]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
vl = {n: float(np.abs(masked[r][LF34][:, t_a]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
vh = {n: float(np.abs(masked[r][HF50][:, t_a]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
print(pd.DataFrame({"a_all_freqs_0-0.5": va, "b_LF": vl, "b_HF": vh, "c_all_freqs_-0.5-0.5": vc}).round(3).to_string())
for lab, v in [("a", va), ("c", vc)]:
    order, top3, tb, k = check_rank(v)
    print(f"reading {lab}: order {order}; named three are the top three {top3} ({k} of 3); temporal below all three {tb}")
ol, tl, tbl, kl = check_rank(vl)
oh, th, tbh, kh = check_rank(vh)
print(f"reading b: LF order {ol}\n           HF order {oh}\n  top three in both {tl and th} (LF {kl}/3, HF {kh}/3); "
      f"temporal below all three in both {tbl and tbh}")

**Result: reproduced.** Under all three readings, postcentral, precentral and inferior parietal are the three regions with the largest mean absolute masked power, and the three temporal regions are below all of them (for example reading a: 0.523, 0.512, 0.336 dB against at most 0.039 for the temporal regions).

## Claim `fig3-lfb-more-widespread`

> Low-frequency power decreases are more spatially widespread than high-frequency increases and
> are also present in supramarginal and superior temporal regions.

In [ ]:
neg_lf = {n: int((masked[r][LF34][:, t_a] < 0).sum()) for r, n in enumerate(ajile.ROI_NAMES)}
pos_hf = {n: int((masked[r][HF50][:, t_a] > 0).sum()) for r, n in enumerate(ajile.ROI_NAMES)}
print(pd.DataFrame({"significant negative 4-30 Hz bins": neg_lf, "significant positive 50-120 Hz bins": pos_hf}).to_string())
a_lf, a_hf = sum(v > 0 for v in neg_lf.values()), sum(v > 0 for v in pos_hf.values())
b_lf, b_hf = sum(neg_lf.values()) / len(LF34), sum(pos_hf.values()) / len(HF50)
print(f"reading a: regions with LF decreases {a_lf}, with HF increases {a_hf}; LF larger {a_lf > a_hf}")
print(f"reading b: LF bins per frequency {b_lf:.2f}, HF bins per frequency {b_hf:.2f}; LF larger {b_lf > b_hf}")
print("supramarginal and superior temporal have LF decreases:", neg_lf["supramarginal"] > 0, neg_lf["superior temporal"] > 0)

**Result: blocked (methods underspecified).** The two readings disagree. Reading a counts 7 regions with low-frequency decreases and 7 with high-frequency increases (no difference); reading b gives 35.2 significant low-frequency bins per frequency against 25.9 high-frequency (low-frequency larger). Supramarginal and superior temporal both have significant 4-30 Hz decreases (30 and 24 bins).

## Claim `fig4-precentral-subject-variability`

> In the precentral region every subject shows the low/high-frequency pattern, but amplitudes
> and frequency bands vary; subjects 03, 06, 07, 08 and 11 have increases up to 120 Hz, subjects
> 09 and 12 mainly at 60-80 Hz, and subjects 04 and 08 have narrower low-frequency decreases.

In [ ]:
pre = REG["precentral"]
sp4 = subj_med[:, pre] - subj_med[:, pre][..., base_i].mean(-1, keepdims=True)     # plot_specs_single_region
spectrum = pd.DataFrame(sp4[..., t_a].mean(-1), index=SUBJECTS, columns=FREQS)
print("Precentral spectrum, mean over 0-0.5 s (dB):")
print(spectrum.round(2).to_string())
lfb_i, hfb_i = (FREQS >= 8) & (FREQS <= 32), (FREQS >= 76) & (FREQS <= 100)
p1 = (spectrum.loc[:, lfb_i].mean(1) < 0) & (spectrum.loc[:, hfb_i].mean(1) > 0)
print("\n(1) LFB mean < 0 and HFB mean > 0:", p1.to_dict(), "->", bool(p1.all()))
p2 = {s: bool(spectrum.loc[s, 116] > 0 and spectrum.loc[s, 121] > 0) for s in ["03", "06", "07", "08", "11"]}
print("(2) 116 and 121 Hz positive:", p2, "->", all(p2.values()))
p3 = {}
for s in ["09", "12"]:
    hi = spectrum.loc[s, FREQS >= 56]
    peak_f = hi.idxmax()
    p3[s] = (int(peak_f), round(hi.max(), 3), round(spectrum.loc[s, (FREQS >= 101)].mean(), 3),
             bool(61 <= peak_f <= 76 and spectrum.loc[s, FREQS >= 101].mean() < hi.max() / 2))
print("(3) peak frequency, peak, mean 101-121 Hz, holds:", p3, "->", all(v[-1] for v in p3.values()))
low = spectrum.loc[:, FREQS <= 46]
width = {s: int((low.loc[s] < low.loc[s].min() / 2).sum()) for s in SUBJECTS}
fewest = sorted(width, key=lambda s: (width[s], s))[:3]
ties = sorted(width.values())[2]
print("(4) width of the decrease (1-46 Hz bins below half the most negative value):", width)
print("    three fewest:", fewest, "; third-smallest width", ties, "shared by",
      [s for s in SUBJECTS if width[s] == ties], "; S04 and S08 among them:", "04" in fewest and "08" in fewest)
# The bin count ties four subjects at 3 bins. Tie-break by the width in Hz of the run of the
# spectrum below half its minimum, around the minimum, with linear interpolation between bins.
def width_hz(v, f):
    m = int(np.argmin(v)); h = v[m] / 2
    lo = m
    while lo > 0 and v[lo - 1] < h:
        lo -= 1
    hi = m
    while hi < len(v) - 1 and v[hi + 1] < h:
        hi += 1
    left = f[lo] if lo == 0 else f[lo - 1] + (h - v[lo - 1]) / (v[lo] - v[lo - 1]) * (f[lo] - f[lo - 1])
    right = f[hi] if hi == len(v) - 1 else f[hi] + (h - v[hi]) / (v[hi + 1] - v[hi]) * (f[hi + 1] - f[hi])
    return right - left
whz = {s: round(width_hz(low.loc[s].values.astype(float), FREQS[FREQS <= 46].astype(float)), 1) for s in SUBJECTS}
fewest_hz = sorted(whz, key=lambda s: whz[s])[:3]
print("    width in Hz (interpolated):", whz, "\n    three narrowest:", fewest_hz,
      "; S04 and S08 among them:", "04" in fewest_hz and "08" in fewest_hz)
print("Parts holding:", int(p1.all()) + int(all(p2.values())) + int(all(v[-1] for v in p3.values())) +
      int("04" in fewest and "08" in fewest), "of 4")

**Result: reproduced.** All four parts hold: every subject has a negative LFB and positive HFB mean in precentral; S03, S06, S07, S08 and S11 are positive at 116 and 121 Hz; S09 and S12 peak at 66 Hz with the 101-121 Hz mean below half the peak; S08 has the narrowest decrease (2 bins, 9.4 Hz). S04 ties with S03, S05 and S07 at 3 bins; by the interpolated width S04 is second narrowest (14.4 Hz), so it is among the three.

## Claim `fig5-band-power-differs-across-subjects`

> Precentral LFB and HFB power in the 0.5 s after movement onset differs significantly across
> subjects.

In [ ]:
E = ev[ev.keep_b]
for meas in ["mean", "median"]:
    for band in ["lfb", "hfb"]:
        col = f"roi_band:precentral:{band}_{meas}"
        h, p = stats.kruskal(*[g[col].values for _, g in E.groupby("subject")])
        print(f"band {meas}, {band.upper()}: Kruskal-Wallis H = {h:.1f}, p = {p:.3g}, n = {len(E)} events in 12 subjects")

**Result: reproduced.** Kruskal-Wallis across the 12 subjects gives p = 1.3e-163 (LFB) and 7.1e-264 (HFB) with the band mean, and p < 1e-150 with the code's median.

## Claim `table1-subject-band-power-ci`

> Each subject's precentral band power lies in the 95% bootstrap interval Table 1 gives; LFB
> power is below baseline and HFB power above it for most subjects.

In [ ]:
TABLE1 = {"01": [(-1.87, -1.29), (0.03, 0.23)], "02": [(-1.12, -0.31), (0.07, 0.64)], "03": [(-2.11, -1.73), (1.39, 1.67)],
          "04": [(-0.17, 0.01), (0.32, 0.52)], "05": [(-0.7, -0.38), (0.2, 0.48)], "06": [(-1.63, -1.11), (2.02, 2.37)],
          "07": [(-0.84, -0.51), (1.14, 1.34)], "08": [(-0.3, 0.04), (0.36, 0.63)], "09": [(-0.57, -0.08), (-0.11, 0.23)],
          "10": [(-0.87, -0.35), (0.01, 0.3)], "11": [(-2.3, -1.86), (0.09, 0.34)], "12": [(-0.66, -0.44), (0.06, 0.2)]}
rng_t1 = np.random.default_rng(1)
rows, inside = [], {}
for sub in SUBJECTS:
    g = E[E.subject == sub]
    for bi_, band in enumerate(["lfb", "hfb"]):
        lo, hi = TABLE1[sub][bi_]
        row = {"subject": sub, "band": band.upper(), "paper_ci": f"{lo} to {hi}"}
        for bm in ["mean", "median"]:
            v = g[f"roi_band:precentral:{band}_{bm}"].values
            for stat_name, fn in [("mean", np.mean), ("median", np.median)]:
                est = fn(v)
                key = f"{stat_name}_of_band{bm}"
                row[key] = round(est, 3)
                inside.setdefault(key, []).append(lo <= est <= hi)
        v = g[f"roi_band:precentral:{band}_mean"].values
        boot = np.array([np.mean(v[rng_t1.integers(0, len(v), len(v))]) for _ in range(5000)])
        row["bootstrap_ci_mean_of_bandmean"] = f"{np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f}"
        rows.append(row)
t1 = pd.DataFrame(rows)
print(t1.to_string(index=False))
print("\nPoint estimates inside the paper's interval, of 24 subject and band pairs:",
      {k: int(sum(v)) for k, v in inside.items()})
lfb_neg = (t1[t1.band == "LFB"]["mean_of_bandmean"] < 0).sum()
hfb_pos = (t1[t1.band == "HFB"]["mean_of_bandmean"] > 0).sum()
print(f"LFB below baseline in {lfb_neg} of 12 subjects; HFB above in {hfb_pos} of 12 (mean of band mean)")

**Result: partially reproduced.** With the mean over events of the band mean, 23 of 24 subject and band values lie inside the paper's intervals; S06 HFB is 1.78 dB against 2.02 to 2.37. The other readings give 20 to 21 of 24. LFB is below baseline and HFB above it in all 12 subjects.

## Claim `fig5-recording-day-effects`

> Precentral band power differs significantly across recording days in LFB for subjects 03,
> 05, 06, 07, 08 and 11, and in HFB for subjects 03, 04, 07, 08 and 10, despite baseline
> subtraction.

In [ ]:
PAPER_SIG = {("03", "lfb"), ("05", "lfb"), ("06", "lfb"), ("07", "lfb"), ("08", "lfb"), ("11", "lfb"),
             ("03", "hfb"), ("04", "hfb"), ("07", "hfb"), ("08", "hfb"), ("10", "hfb")}
rows = []
for meas in ["mean", "median"]:
    for band in ["lfb", "hfb"]:
        for sub in SUBJECTS:
            g = E[E.subject == sub]
            h, p = stats.kruskal(*[d[f"roi_band:precentral:{band}_{meas}"].values for _, d in g.groupby("day")])
            rows.append({"measure": meas, "band": band.upper(), "subject": sub, "p": p, "significant": p < 0.05,
                         "paper_significant": (sub, band) in PAPER_SIG})
dayk = pd.DataFrame(rows)
dayk["agrees"] = dayk.significant == dayk.paper_significant
print(dayk.pivot_table(index=["band", "subject"], columns="measure", values="p").round(4).to_string())
print("\nAgreement with the paper's 24 subject and band pairs:", dayk.groupby("measure").agrees.sum().to_dict())
for meas in ["mean", "median"]:
    d = dayk[dayk.measure == meas]
    print(f"band {meas}: significant here but not in the paper {d[d.significant & ~d.paper_significant][['band','subject']].values.tolist()}; "
          f"in the paper but not here {d[~d.significant & d.paper_significant][['band','subject']].values.tolist()}")

**Result: reproduced.** With the band mean, the Kruskal-Wallis tests across days agree with the paper on all 24 subject and band pairs (the 11 significant pairs and the 13 others); with the code's band median, 23 of 24 (LFB S05 gives p = 0.088).

## Claim `extfig5-1-baseline-reduces-day-variability`

> Subtracting the -1.5 to -1 s baseline reduces the variability of band power across recording
> days, though clear variability remains.

In [ ]:
# Electrode band power (band mean), with and without baseline subtraction, for kept events.
el = {}
i = 0
for s, r in zip(sessions, ecog):
    for k, ok in enumerate(r["ok"]):
        if keep[i]:
            for kind in ["el_band", "el_band_raw"]:
                for band in ["lfb", "hfb"]:
                    el.setdefault((s["subject"], kind, band), []).append((s["day"], r[kind][k][f"{band}_mean"]))
        i += 1
rows_a, rows_b = [], []
for sub in SUBJECTS:
    good = first_good[sub]
    for band in ["lfb", "hfb"]:
        g = E[E.subject == sub]
        sd = {}
        for kind, col in [("with", f"roi_band:precentral:{band}_mean"), ("without", f"roi_band_raw:precentral:{band}_mean")]:
            sd[kind] = g.groupby("day")[col].mean().std()
        rows_a.append({"subject": sub, "band": band.upper(), "sd_with": round(sd["with"], 3), "sd_without": round(sd["without"], 3)})
        for kind in ["with", "without"]:
            items = el[(sub, "el_band" if kind == "with" else "el_band_raw", band)]
            days_ = np.array([d for d, _ in items])
            vals = np.array([v for _, v in items])[:, good]
            dm = np.array([vals[days_ == d].mean(0) for d in np.unique(days_)])
            sd[kind + "_el"] = dm.std(0, ddof=1)
        for k_ in range(good.sum()):
            rows_b.append({"subject": sub, "band": band.upper(), "sd_with": sd["with_el"][k_], "sd_without": sd["without_el"][k_]})
ra, rb = pd.DataFrame(rows_a), pd.DataFrame(rows_b)
print("Reading a, precentral region per subject (SD across days of day means, dB):")
print(ra.pivot(index="subject", columns="band", values=["sd_with", "sd_without"]).to_string())
for band in ["LFB", "HFB"]:
    a_ = ra[ra.band == band]
    b_ = rb[rb.band == band]
    print(f"{band}: reading a, SD lower with baseline in {(a_.sd_with < a_.sd_without).sum()} of {len(a_)} subjects "
          f"(median {a_.sd_with.median():.3f} against {a_.sd_without.median():.3f} dB); reading b, in "
          f"{(b_.sd_with < b_.sd_without).sum()} of {len(b_)} good electrodes ({(b_.sd_with < b_.sd_without).mean():.3f}; "
          f"median {b_.sd_with.median():.3f} against {b_.sd_without.median():.3f} dB)")
print("Day effects that remain (claim fig5-recording-day-effects, band mean): significant subject and band pairs",
      int(dayk[(dayk.measure == "mean") & dayk.significant].shape[0]))

**Result: reproduced.** The SD across days is lower with baseline subtraction for 9 of 12 subjects in LFB and 12 of 12 in HFB (reading a) and for 0.918 and 0.901 of good electrodes (reading b). Variability remains: 11 subject and band pairs have significant day effects after baseline subtraction (median SD with baseline 0.20 and 0.14 dB in the precentral region).

## Claim `discussion-day-changes-small`

> Day-to-day changes in baseline-subtracted spectral power are small (about 1-2 dB) compared
> with spectral power without baseline subtraction (about 10-50 dB).

In [ ]:
rows = []
for sub in SUBJECTS:
    g = E[E.subject == sub]
    for band in ["lfb", "hfb"]:
        col = f"roi_band:precentral:{band}_mean"
        dm = g.groupby("day")[col].mean()
        raw = g[f"roi_band_raw:precentral:{band}_mean"].mean()
        rows.append({"subject": sub, "band": band.upper(), "largest_day_change_dB": round((dm - g[col].mean()).abs().max(), 3),
                     "day_mean_range_dB": round(dm.max() - dm.min(), 3), "level_uV_dB": round(raw, 2),
                     "level_V_dB": round(raw - 120, 2)})
dc = pd.DataFrame(rows)
print(dc.to_string(index=False))
print(f"\nlargest day change {dc.largest_day_change_dB.max():.3f} dB (holds at <= 2.2: {(dc.largest_day_change_dB <= 2.2).all()}); "
      f"range of day means up to {dc.day_mean_range_dB.max():.3f} dB")
print(f"levels without baseline: microvolts {dc.level_uV_dB.min():.1f} to {dc.level_uV_dB.max():.1f} dB "
      f"(within 9-55: {dc.level_uV_dB.between(9, 55).all()}); volts {dc.level_V_dB.min():.1f} to {dc.level_V_dB.max():.1f} dB "
      f"(within 9-55: {dc.level_V_dB.between(9, 55).all()})")

**Result: reproduced.** The largest deviation of a day mean from the subject mean is 1.01 dB (the range of day means reaches 1.80 dB), and the power without baseline subtraction is 12.4 to 36.3 dB with the data in microvolts, inside 9 to 55 dB; in volts it would be -107.5 to -83.7 dB.

## Regression claims (Figure 6 and Extended Data Figure 6-2): timing

The six regression claims need, for every good electrode of every subject and both bands, forward
selection and 200 day-balanced splits, each with a Huber fit of the full model and one refit per
selected feature (opening comment of the claims file). One model is timed here and the whole is
estimated from it.

In [ ]:
rng_fig6 = np.random.default_rng(6)
sub = "07"
g = E[E.subject == sub]
X, groups = ajile.regression_design(g)
items = el[(sub, "el_band", "lfb")]
vals = np.array([v for _, v in items])
best = int(np.argmax([abs(np.corrcoef(X["angle"], vals[:, k])[0, 1]) for k in range(vals.shape[1])]))
t0 = time.time()
model = ajile.regression_model(X, groups, vals[:, best], g.day.values, rng_fig6, n_splits=200)
t_model = time.time() - t0
n_models = 2 * sum(int(first_good[s].sum()) for s in SUBJECTS)
print(f"one model (S{sub}, electrode {best}, LFB, {len(g)} events, 200 splits): {t_model:.1f} s; "
      f"selected {model['selected']}; withheld R2 {model['r2']:.3f}")
print(f"models needed: {n_models} (good electrodes x 2 bands); estimate for the main analysis "
      f"{t_model * n_models / 3600:.1f} h on one core, {t_model * n_models / 3600 / os.cpu_count():.1f} h on "
      f"{os.cpu_count()} cores; the claims add three variants (80/20 split, the code's shuffle, bad electrodes)")
r2_file = pd.concat([s["electrodes"].assign(subject=s["subject"]) for s in sessions if s["day"] == min(
    x["day"] for x in sessions if x["subject"] == s["subject"])])
print("For reference, the regression R2 stored in the files' electrode tables (low_freq_R2, high_freq_R2): "
      f"max {r2_file.low_freq_R2.max():.3f} and {r2_file.high_freq_R2.max():.3f}; electrodes with values "
      f"{int(r2_file.low_freq_R2.notna().sum())} of {len(r2_file)}")

**Result: blocked (too large).** One model takes the time printed above; the 2192 models of the main analysis alone exceed the notebook limit of 60 minutes on this machine, before the variants the claims ask for. The six regression claims are blocked for this reason.

In [ ]:
# How much the assumed reading of the bad-segment rule matters: the main ECoG numbers with only
# the segments with missing samples dropped (no 0 dB or 115-125 Hz rule).
E_alt = ev[ev.ok & ~ev.bad_missing]
print("events:", len(E_alt), "against", len(E))
for band in ["lfb", "hfb"]:
    col = f"roi_band:precentral:{band}_mean"
    h, p = stats.kruskal(*[g[col].values for _, g in E_alt.groupby("subject")])
    ins = sum(TABLE1[s][0 if band == "lfb" else 1][0] <= E_alt[E_alt.subject == s][col].mean() <= TABLE1[s][0 if band == "lfb" else 1][1]
              for s in SUBJECTS)
    sig = [s for s in SUBJECTS if stats.kruskal(*[d[col].values for _, d in E_alt[E_alt.subject == s].groupby("day")])[1] < 0.05]
    print(f"{band.upper()}: across subjects p = {p:.3g}; Table 1 means inside {ins} of 12; day effects in {sig}")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| subject-days (link) | reproduced |
| roi-projection-link (link, Figshare projection matrices) | reproduced |
| cohort | partially reproduced (seven subjects with depth electrodes, not five; sex and age not in the dataset) |
| fig2-event-counts | not reproduced (211 to 996 events per subject, median 770.5) |
| fig2-features-differ-across-subjects | reproduced |
| fig2-reach-shape | partially reproduced (angle peaks near +-70, not +-90) |
| fig2-bimanual-skew | reproduced |
| extfig2-1c-contralateral-larger | reproduced |
| extfig2-2-feature-correlations | partially reproduced (5 of 6) |
| fig2-days-and-waking | partially reproduced (S04's busiest day holds 38%) |
| fig3-group-lfb-decrease-hfb-increase | reproduced |
| fig3-sensorimotor-largest | reproduced |
| fig3-lfb-more-widespread | blocked, methods underspecified (readings disagree) |
| fig4-precentral-subject-variability | reproduced (one tie broken by interpolated width) |
| fig5-band-power-differs-across-subjects | reproduced |
| table1-subject-band-power-ci | partially reproduced (23 of 24) |
| fig5-recording-day-effects | reproduced (24 of 24) |
| extfig5-1-baseline-reduces-day-variability | reproduced |
| discussion-day-changes-small | reproduced |
| six regression claims (Figure 6, Extended Data Figure 6-2) | blocked, too large |

**How the analysis differs from the paper's.** The paper's preprocessing, event pruning and visual
removal of false positives were done before the data were deposited; the dataset holds 8088 events and
marks none as false positives, so all are kept. The bad-segment rules use an assumed summary of
segment power; with only segments with missing samples dropped, the across-subject tests, the day
effects and the Table 1 outcome (22 of 24 inside) do not change (last data check above). Region power uses the paper's Figshare projection matrices, not a new
projection from the AAL atlas. Bad electrodes are those of each subject's first file. The Morlet
transform is computed by FFT and matches MNE's direct convolution. The regression claims were not run.

**What was found about the dataset.** (1) In three files (S01 day 7, S04 day 3, S06 day 3) the ECoG
covers only the end of the day but its `starting_time` is 0, while the event table and pose use a
clock from midnight; the ECoG must be shifted by 8.1 to 8.5 h, as `ReachEvents` and the neural
response show. (2) The files hold 8088 events; the data descriptor states 6931 verified events, and
the paper's counts per subject are lower than the files'. (3) The reach angle is stored in image
coordinates (upward negative) and unfolded, without saying so. (4) The pose is labelled meters but
is in pixels. (5) `session_start_time` gives midnight UTC labelled as 19:00 at UTC-5. (6) Bimanual
overlap is described as a fraction but reaches 3.83 (it is in seconds). (7) 124 event segments contain
missing ECoG samples. (8) Sex and age are not recorded. (9) Seven subjects have electrode groups
described as depth, where the paper says five.